# Part 1: Partial Freezing of Distil-mBERT for PoS Tagging

This notebook wraps the reproducible experiment implemented in `scripts/run_pos_freezing_experiment.py`. It fine-tunes `distilbert-base-multilingual-cased` on a subset of UD English-EWT and compares full fine-tuning against freezing embeddings plus the first three DistilBERT transformer layers.

In [ ]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## Run the Experiment

The script downloads the UD English-EWT CoNLL-U files if needed, tokenizes the selected subset, trains both model variants, and writes JSON/CSV results under `outputs/pos_freezing`.

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    'scripts/run_pos_freezing_experiment.py',
    '--max-train-samples', '3000',
    '--max-eval-samples', '1000',
    '--epochs', '3',
    '--batch-size', '16',
    '--output-dir', 'outputs/pos_freezing',
], check=True)

## Inspect the Results

In [ ]:
import pandas as pd

summary = pd.read_csv('outputs/pos_freezing/summary.csv')
summary

In [ ]:
import json
from pathlib import Path

results = json.loads(Path('outputs/pos_freezing/results.json').read_text(encoding='utf-8'))
results['dataset_sizes'], results['cuda_device'], results['labels']

## Summary

In the completed run, full fine-tuning reached 95.79% test accuracy. Partial freezing reached 94.92% test accuracy while reducing trainable parameters from 134.7M to 21.3M and reducing runtime from 1.37 minutes to 0.77 minutes. This suggests a clear compute/performance tradeoff: full fine-tuning gives the best accuracy, but partial freezing preserves most performance with much lower update cost.